# Study Tutor — the agents

Built on the RAG pipeline from `tutor.ipynb`. Run that first: this notebook needs
an indexed document.

Four specialists and an orchestrator that routes between them. Each is a prompt in
`prompts/`, a Pydantic schema, and its own retrieval strategy.

| Agent | Retrieval |
|---|---|
| Answerer | on the question |
| Topic Extractor | none — needs the whole document |
| Exam Generator | on the topic and its subtopics |
| Grader | on the question **and** the student's answer |


## 0. Setup


In [ ]:
# Pick up edits to tutor/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

# The notebook lives in notebooks/, the package one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from tutor import config   # importing this creates .env from .env.example if missing

# autoreload watches .py files, not .env - settings are re-read explicitly.
config.reload()

# Silence the SDK's automatic-function-calling notice; we use no tools.
logging.getLogger('google_genai.models').setLevel(logging.ERROR)

print('repo root      ', config.ROOT_DIR)
print('chat model     ', config.GEMINI_MODEL)
print('embed model    ', config.GEMINI_EMBED_MODEL, f'({config.EMBED_DIM} dims)')
print('chunk size     ', config.CHUNK_SIZE, 'chars, overlap', config.CHUNK_OVERLAP)
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off (Gemini only)')
print('PDFs in data/  ', [p.name for p in config.DATA_DIR.glob('*.pdf')] or 'NONE - add one!')

# Fail here, not 20 cells later with a 404 that blames the model.
try:
    config.validate_models()
except Exception as error:
    print('\nCONFIGURATION PROBLEM:\n')
    print(error)

if config.GOOGLE_API_KEY:
    print('API key        loaded OK')
else:
    print(f'API key        MISSING -> open {config.ENV_FILE} and paste your key')


## 1. Topic Extractor

Returns the topics, subtopics and pages of the document.

**Does not use RAG.** Extracting topics needs coverage of the whole document, and
retrieval is built to discard most of it. Opposite goals.

Map-reduce: topics per batch, then one call merging duplicates. With a single batch the
merge is skipped.


In [ ]:
from tutor import agents, prompts

# The agents live in tutor/agents.py so the CLI (scripts/chat.py) can use the same
# code this notebook demonstrates. Their prompts are files in prompts/.
print(agents.TopicMap.model_json_schema()['$defs']['Topic']['properties'].keys())
print()
print(prompts.load('topic_extractor'))


Every agent runs on the shared persona plus its own role, composed in
`prompts.system()`, so none can run ungrounded by accident.


In [ ]:
# Every agent runs on the shared persona + its own role. Composing them means no
# agent can run ungrounded by accident.
print(prompts.system(prompts.load('topic_extractor'))[:600], '...')


### The agent

Batches are sized in characters, not chunks. Each passage is labelled `[page N]` — the
only honest source for the page numbers.


In [ ]:
import inspect

print(inspect.getsource(agents.extract_topics))


In [ ]:
topic_map = agents.extract_topics()

for topic in topic_map.topics:
    pages = ', '.join(str(p) for p in sorted(set(topic.source_pages)))
    print(f'\n{topic.name}   (p. {pages})')
    print(f'  {topic.summary}')
    for subtopic in topic.subtopics:
        print(f'    - {subtopic}')


### Check it against your PDF

Wrong pages mean the `[page N]` markers are being ignored, and every later citation is
fiction. A missing topic will never be examined.


## 2. Exam Generator

RAG on the topic name **plus its subtopics** — the name alone retrieves too narrowly for
four different questions.

Each question carries `key_points`: what a correct answer must contain. The generator
had the passage in front of it, so it is the cheapest place to decide that.


In [ ]:
print(prompts.load('exam_generator'))
print()
print('Exam question fields:', list(agents.ExamQuestion.model_fields))


### Verification

The prompt asks for a verbatim quote. Asking is not enough — a model under pressure
produces something quote-shaped. So every quote is checked against its source passage.


In [ ]:
# Pick any topic from the map above by index.
TOPIC_INDEX = 0
topic = topic_map.topics[TOPIC_INDEX]
print('Topic:', topic.name, '\n')

exam, hits = agents.generate_exam(topic, n_questions=4)
report = agents.verify_exam(exam, hits)

for q, check in zip(exam.questions, report):
    flag = 'OK  ' if check['grounded'] else 'FLAG'
    print(f"\n[{flag}] Q{check['n']} ({q.difficulty}, p.{q.source_page}, "
          f"quote match {check['score']:.0%})")
    print(' ', q.question)
    print('   key points:', '; '.join(q.key_points))
    if not check['grounded']:
        print('   UNVERIFIED QUOTE:', q.source_quote[:160])

passed = sum(c['grounded'] and c['page_ok'] for c in report)
print(f'\n{passed}/{len(report)} questions traced to the document.')


### Reading the flags

85–100% grounded · 50–85% probably tidied up, check it · under 50% not in the document,
discard.


## 3. Grader

Walks the `key_points` and marks each `covered` / `partial` / `missing`. Retrieval runs
on the question **and** the student's answer, so unsupported claims can be identified.

**The score is computed in code, never asked for.** A model asked for "a score out of
10" is not reproducible and often contradicts its own feedback.


In [ ]:
print(prompts.load('grader'))
print()
print('Feedback fields:', list(agents.RawFeedback.model_fields), '- note: no score')


In [ ]:
from tutor.scoring import score_from_points, verdict_from_score

# The score is derived from the per-criterion verdicts, not asked from the model.
for statuses in (['covered'] * 3,
                 ['covered', 'covered', 'partial'],
                 ['covered', 'missing', 'missing']):
    score = score_from_points(statuses)
    print(f'{str(statuses):48} {score:.0%}  {verdict_from_score(score)}')


### Three answers

Good, half-right, and confidently wrong. The third is the real test.


In [ ]:
question = exam.questions[0]
print('Q:', question.question)
print('Key points:', '; '.join(question.key_points), '\n')

# Built from the document itself, so this works with any PDF:
#  - good    : the reference answer
#  - partial : only its first sentence
#  - wrong   : fluent prose about a DIFFERENT topic - plausible, and not an answer
other = next((t for t in topic_map.topics if t.name != topic.name), topic)
ANSWERS = {
    'good': question.expected_answer,
    'partial': question.expected_answer.split('.')[0] + '.',
    'confidently wrong': other.summary,
}

for label, answer in ANSWERS.items():
    print('\n' + '=' * 70)
    print(f'ANSWER ({label}): {answer[:110]}...')
    print('=' * 70)
    print(agents.format_feedback(agents.grade_answer(question, answer)))


### What to check

The wrong answer must come out INCORRECT. The percentage must match the `+ ~ -` marks —
it is computed from them.


## 4. Orchestrator

Classifies the message and delegates. **Never answers from the document itself** — every
grounding guarantee lives inside the specialists.

Routing is structured output: a `Literal` enforced during decoding, so an invalid route
cannot be returned. The topic name it extracts is matched against the real topic map
before reaching retrieval.


In [ ]:
from tutor import orchestrator

print(prompts.load('router'))
print()
print('Route fields:', list(orchestrator.Route.model_fields))


### Dispatch and memory

Sliding window of 6 turns plus a rolling summary. Structured state (open exam, scores)
lives in `StudySession`, never in the transcript — which is what makes the window safe.


In [ ]:
print(inspect.getsource(orchestrator.handle))


### Full conversation

Watch the `[route: ...]` tag. Message 4 is a bare answer with no keywords and still
reaches the grader, because a question is open.


In [ ]:
from tutor.session import StudySession

session = StudySession(topic_map=topic_map)

# Built from the actual topics, so it works with any document.
SCRIPT = [
    '¿De qué trata este documento?',
    f'Explícame el tema: {topic.name}',
    f'Hazme 2 preguntas de práctica sobre {topic.name}',
    ANSWERS['partial'],
    '¿Cómo voy?',
    '¿Cuál es la capital de Francia?',
]

for message in SCRIPT:
    print('\n' + '=' * 74)
    print('STUDENT:', message)
    print('-' * 74)
    print(orchestrator.handle(message, session))


## 5. Try it yourself

Ask anything, answer an open question, `salir` to stop. Same loop as
`python scripts/chat.py`.


In [ ]:
session = StudySession(topic_map=topic_map)

print('Ask about your document. Type "salir" to stop.\n')
while True:
    message = input('> ').strip()
    if not message or message.lower() in {'salir', 'exit', 'quit'}:
        break
    print('\n' + orchestrator.handle(message, session) + '\n')

for name, average in session.weakest_topics():
    print(f'{average:.0%}  {name}')
